<a href="https://colab.research.google.com/github/Rushikesh042/AI-Fitness-Coach/blob/main/DATASET_PREPROCESSING_(MMDental_%26_ToothFairy2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CONTENT IN THE DATASET**

In [ ]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 17.2 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# **DATASET PRE-PROCESSING (MMDental Dataset)**

In [ ]:
# -*- coding: utf-8 -*-
"""
MMDental Preprocessing Pipeline  —  v3
======================================

Covers:
  1. 3D CBCT volume loading, orientation, resampling
  2. Multichannel montage generation (axial / coronal / sagittal × slice / MIP / mean)
  3. CSV cleaning, label extraction, chief-complaint prioritisation
  4. Per-image VQA record generation for VLM instruction tuning
  5. Class imbalance analysis and WeightedRandomSampler weights
  6. Patient-level primary-label stratified train / val / test split with leakage check
  7. JSONL output in Qwen2.5-VL typed-content format

Usage:
  pip install SimpleITK nibabel opencv-python-headless tqdm
  python mmdental_pipeline_v3.py
"""

import collections
import glob
import hashlib
import json
import os
import random
import re
import warnings
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import SimpleITK as sitk

warnings.filterwarnings("ignore")


DATASET_PATH = "/content/drive/MyDrive/Dental_Project/MMDental"
CSV_PATH = "/content/drive/MyDrive/Dental_Project/MMDental/medical_records.csv"
OUT_DIR = "/content/drive/MyDrive/Dental_Project/MMDental_Preprocessed"

IMG_DIR = os.path.join(OUT_DIR, "images")
DEBUG_DIR = os.path.join(OUT_DIR, "debug_arch_slices")
SPLIT_DIR = os.path.join(OUT_DIR, "splits")
STATS_OUT = os.path.join(OUT_DIR, "dataset_stats.json")
JSONL_SOURCE = os.path.join(OUT_DIR, "source.jsonl")

for _d in [OUT_DIR, IMG_DIR, DEBUG_DIR, SPLIT_DIR]:
    os.makedirs(_d, exist_ok=True)

ISOTROPIC_SPACING: Tuple[float, float, float] = (0.4, 0.4, 0.4)

TILE_HW = (448, 448)
NUM_Z_OFFSETS = 3
Z_DELTA_FRAC = 0.10
Y_SHIFT_FRAC = 0.03
X_SHIFT_FRAC = 0.03
SLAB_FRAC = 0.07
MIP_CAP_P = 99.0

CRANIAL_FRAC: Optional[float] = 0.40
SAVE_DEBUG_ARCH_SLICES = True

LABEL_SPACE = ["ENDO", "PERIO", "DEV_POS", "CARIES_STRUCT"]

TASK_DESCRIBE = "describe"
TASK_CLASSIFY = "classify"
TASK_TOOTH = "tooth_specific"
TASK_TREATMENT = "treatment"
TASK_HISTORY = "history"
ENABLED_TASKS = [TASK_DESCRIBE, TASK_CLASSIFY, TASK_TOOTH, TASK_TREATMENT, TASK_HISTORY]
CSV_TEXT_TASKS = {TASK_DESCRIBE, TASK_TOOTH, TASK_TREATMENT, TASK_HISTORY}

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

KEEP_NEGATIVES = True

MIN_PIXELS = 256 * 256
MAX_PIXELS = 1344 * 1344

_DIGITS_RE = re.compile(r"(\d+)$")


def safe_str(x: Any) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and np.isnan(x):
            return ""
    except TypeError:
        pass
    s = str(x).strip()
    return "" if s.lower() == "nan" else s


def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def write_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(to_jsonable(r), ensure_ascii=False) + "\n")


def sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _canon_id(s: Any) -> str:
    s = str(s).strip()
    m = _DIGITS_RE.search(s)
    return m.group(1) if m else s


def _id_variants(id_str: str, pad_widths: Tuple[int, ...] = (3, 4, 5, 6)) -> List[str]:
    v, s = set(), _canon_id(id_str)
    v.add(s)
    try:
        n = int(s)
        v.add(str(n))
        for w in pad_widths:
            v.add(str(n).zfill(w))
    except ValueError:
        pass
    return sorted(v)


def _linspace_offsets(n: int) -> List[float]:
    if n <= 1:
        return [0.0]
    return np.linspace(-1.0, 1.0, n).tolist()


def build_volume_index(dataset_root: str) -> Dict[str, Tuple[str, str]]:
    index: Dict[str, Tuple[str, str]] = {}

    nifti_paths = glob.glob(os.path.join(dataset_root, "**", "*.nii"), recursive=True)
    nifti_paths += glob.glob(os.path.join(dataset_root, "**", "*.nii.gz"), recursive=True)

    for p in nifti_paths:
        stem = os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")
        for key in _id_variants(stem):
            existing = index.get(key)
            if existing is None or (
                p.endswith(".nii.gz") and existing[0] == "nifti" and not existing[1].endswith(".nii.gz")
            ):
                index[key] = ("nifti", p)

    dcm_dirs = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(dataset_root, "**", "*.dcm"), recursive=True)})
    for d in dcm_dirs:
        for key in _id_variants(os.path.basename(d)):
            if key not in index:
                index[key] = ("dicom", d)

    return index


def find_volume(pid: str, vol_index: Dict[str, Tuple[str, str]]) -> Optional[Tuple[str, str]]:
    for v in _id_variants(pid):
        if v in vol_index:
            return vol_index[v]
    return None


def _read_sitk(kind: str, path: str) -> sitk.Image:
    if kind == "nifti":
        return sitk.ReadImage(path)
    if kind == "dicom":
        reader = sitk.ImageSeriesReader()
        ids = reader.GetGDCMSeriesIDs(path)
        if not ids:
            raise FileNotFoundError(f"No DICOM series in {path}")
        reader.SetFileNames(reader.GetGDCMSeriesFileNames(path, ids[0]))
        return reader.Execute()
    raise ValueError(f"Unknown kind: {kind}")


def load_volume(kind: str, path: str) -> np.ndarray:
    img = _read_sitk(kind, path)
    try:
        img = sitk.DICOMOrient(img, "LPS")
    except Exception:
        pass

    sp = img.GetSpacing()
    sz = img.GetSize()
    out_sz = [max(1, int(round(sz[i] * sp[i] / ISOTROPIC_SPACING[i]))) for i in range(3)]

    rs = sitk.ResampleImageFilter()
    rs.SetOutputSpacing(ISOTROPIC_SPACING)
    rs.SetSize(out_sz)
    rs.SetOutputDirection(img.GetDirection())
    rs.SetOutputOrigin(img.GetOrigin())
    rs.SetInterpolator(sitk.sitkLinear)
    rs.SetDefaultPixelValue(0)

    return sitk.GetArrayFromImage(rs.Execute(img)).astype(np.float32)


def is_hu_like(vol: np.ndarray) -> bool:
    p1 = float(np.nanpercentile(vol, 1))
    p99 = float(np.nanpercentile(vol, 99))
    vmin, vmax = float(np.nanmin(vol)), float(np.nanmax(vol))
    return bool(
        ((p1 < -200) and (p99 > 800))
        or ((vmin < -500) and (vmax > 1500))
        or ((p99 - p1) > 1000)
        or ((vmax - vmin) > 2000 and vmax > 1000)
    )


def window_norm(vol: np.ndarray, level: float, width: float) -> np.ndarray:
    lo, hi = level - width / 2.0, level + width / 2.0
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0.0, 1.0)


def robust_norm(vol: np.ndarray, p_lo: float = 1.0, p_hi: float = 99.0) -> np.ndarray:
    lo = float(np.nanpercentile(vol, p_lo))
    hi = float(np.nanpercentile(vol, p_hi))
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0.0, 1.0)


def build_rgb_channels(vol: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if is_hu_like(vol):
        v_r = window_norm(vol, 400.0, 1500.0)
        v_g = window_norm(vol, 40.0, 400.0)
        v_b = window_norm(vol, 1500.0, 2000.0)
    else:
        v_r = robust_norm(vol, 1.0, 97.0)
        v_g = robust_norm(vol, 20.0, 80.0)
        v_b = robust_norm(vol, 75.0, 99.9)
    return v_r, v_g, v_b


def detect_arch_z(vol: np.ndarray,
                  border_frac: float = 0.12,
                  bone_thr_p: float = 85.0,
                  pid: str = "",
                  debug_dir: Optional[str] = None) -> int:
    Z, Y, X = vol.shape
    v = robust_norm(vol, 1, 99)

    by = int(Y * border_frac)
    bx = int(X * border_frac)
    v_roi = v[:, by:Y - by, bx:X - bx] if (Y - 2 * by > 10 and X - 2 * bx > 10) else v
    v_roi = np.minimum(v_roi, float(np.nanpercentile(v_roi, 99.5)))

    thr = float(np.nanpercentile(v_roi, bone_thr_p))
    mask = (v_roi > thr).astype(np.float32)
    yy = mask.shape[1]
    mask *= np.linspace(0.7, 1.3, yy, dtype=np.float32)[None, :, None]

    score = mask.sum(axis=(1, 2))

    if CRANIAL_FRAC is not None:
        # For this dataset after LPS orientation + GetArrayFromImage:
        # z=0 is inferior and z=Z-1 is superior.
        # Suppress the superior cranial end, keep the inferior dental end.
        lower_limit = int(Z * (1.0 - CRANIAL_FRAC))
        score[lower_limit:] = 0.0

    k = max(5, int(Z * 0.03)) | 1
    pad = k // 2
    sm = np.convolve(np.pad(score, (pad, pad), "edge"), np.ones(k, np.float32) / k, "valid")
    z0 = int(np.argmax(sm))

    if SAVE_DEBUG_ARCH_SLICES and debug_dir and pid:
        slice_img = np.clip(robust_norm(vol[z0]) * 255, 0, 255).astype(np.uint8)
        out_path = os.path.join(debug_dir, f"{pid}_arch_z{z0:03d}.png")
        cv2.imwrite(out_path, slice_img)

    return z0


def compute_yx_roi(
    v_norm: np.ndarray,
    z0: int,
    slab_frac: float = 0.05,
    pad_frac: float = 0.02,
    border_frac: float = 0.12,
    thr_p: float = 60.0,
) -> Tuple[int, int, int, int]:
    Z, Y, X = v_norm.shape
    slab = max(8, int(Z * slab_frac))
    proj = np.mean(v_norm[max(0, z0 - slab): min(Z, z0 + slab + 1)], axis=0)

    thr = float(np.nanpercentile(proj, thr_p))
    mask = (proj > thr).astype(np.uint8)
    by, bx = int(Y * border_frac), int(X * border_frac)

    if Y - 2 * by > 10 and X - 2 * bx > 10:
        mask[:by, :] = mask[Y - by:, :] = 0
        mask[:, :bx] = mask[:, X - bx:] = 0

    n_lbl, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n_lbl <= 1:
        cy, cx = Y // 2, X // 2
        h = min(Y, X) // 3
        return max(0, cy - h), min(Y, cy + h), max(0, cx - h), min(X, cx + h)

    k = int(np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1)
    x, y, w, h, _ = stats[k]
    py = max(10, int(h * pad_frac))
    px = max(10, int(w * pad_frac))
    return max(0, y - py), min(Y, y + h + py), max(0, x - px), min(X, x + w + px)


def _capped_mip(stack: np.ndarray, axis: int = 0) -> np.ndarray:
    cap = float(np.nanpercentile(stack, MIP_CAP_P))
    return np.max(np.minimum(stack, cap), axis=axis)


def _aspect_resize(img: np.ndarray, out_hw: Tuple[int, int]) -> np.ndarray:
    oh, ow = out_hw
    h, w = img.shape
    scale = min(ow / max(w, 1), oh / max(h, 1))
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))
    u8 = np.clip(img * 255.0, 0, 255).astype(np.uint8)
    u8 = cv2.resize(u8, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((oh, ow), dtype=np.uint8)
    canvas[(oh - nh) // 2:(oh - nh) // 2 + nh, (ow - nw) // 2:(ow - nw) // 2 + nw] = u8
    return canvas.astype(np.float32) / 255.0


def extract_views(v: np.ndarray, z: int, y: int, x: int) -> Dict[str, np.ndarray]:
    Z, Y, X = v.shape
    z = int(np.clip(z, 0, Z - 1))
    y = int(np.clip(y, 0, Y - 1))
    x = int(np.clip(x, 0, X - 1))

    sz = max(12, int(Z * SLAB_FRAC))
    sy = max(12, int(Y * SLAB_FRAC))
    sx = max(12, int(X * SLAB_FRAC))

    def _proj(data: np.ndarray, idx: int, slab: int, axis: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        sl = np.take(data, np.arange(max(0, idx - slab), min(data.shape[axis], idx + slab + 1)), axis=axis)
        return (
            np.take(data, idx, axis=axis),
            _capped_mip(sl, axis=axis),
            np.mean(sl, axis=axis),
        )

    ax_s, ax_m, ax_mn = _proj(v, z, sz, 0)
    cor_s, cor_m, cor_mn = _proj(v, y, sy, 1)
    sag_s, sag_m, sag_mn = _proj(v, x, sx, 2)

    cor_s, cor_m, cor_mn = [np.rot90(i, 1) for i in (cor_s, cor_m, cor_mn)]
    sag_s, sag_m, sag_mn = [np.rot90(i, 1) for i in (sag_s, sag_m, sag_mn)]

    return {
        "axial": ax_s,
        "axial_mip": ax_m,
        "axial_mean": ax_mn,
        "coronal": cor_s,
        "coronal_mip": cor_m,
        "coronal_mean": cor_mn,
        "sagittal": sag_s,
        "sagittal_mip": sag_m,
        "sagittal_mean": sag_mn,
    }


def make_montage(v_r: np.ndarray, v_g: np.ndarray, v_b: np.ndarray, z: int, y: int, x: int) -> Image.Image:
    vr = extract_views(v_r, z, y, x)
    vg = extract_views(v_g, z, y, x)
    vb = extract_views(v_b, z, y, x)

    row_keys = [
        ["axial", "axial_mip", "axial_mean"],
        ["coronal", "coronal_mip", "coronal_mean"],
        ["sagittal", "sagittal_mip", "sagittal_mean"],
    ]

    def _grid(views: Dict[str, np.ndarray]) -> np.ndarray:
        return np.vstack([np.hstack([_aspect_resize(views[k], TILE_HW) for k in rk]) for rk in row_keys])

    rgb = np.stack(
        [
            np.clip(_grid(vr) * 255, 0, 255).astype(np.uint8),
            np.clip(_grid(vg) * 255, 0, 255).astype(np.uint8),
            np.clip(_grid(vb) * 255, 0, 255).astype(np.uint8),
        ],
        axis=-1,
    )
    return Image.fromarray(rgb, "RGB")


def create_patient_images(pid: str, kind: str, vol_path: str) -> Optional[List[str]]:
    try:
        vol = load_volume(kind, vol_path)
        if vol.ndim != 3 or min(vol.shape) < 16:
            raise ValueError(f"Bad shape {vol.shape}")
    except Exception as e:
        print(f"  [WARN] {pid}: volume load failed — {e}")
        return None

    z0_full = detect_arch_z(vol, pid=pid, debug_dir=DEBUG_DIR if SAVE_DEBUG_ARCH_SLICES else None)

    v_loc = window_norm(vol, 400, 1500) if is_hu_like(vol) else robust_norm(vol, 1, 99)
    y1, y2, x1, x2 = compute_yx_roi(v_loc, z0_full)
    vol_crop = vol[:, y1:y2, x1:x2]

    z0 = detect_arch_z(vol_crop, pid=f"{pid}_crop", debug_dir=DEBUG_DIR if SAVE_DEBUG_ARCH_SLICES else None)
    v_r, v_g, v_b = build_rgb_channels(vol_crop)

    Z, Y, X = vol_crop.shape
    delta = max(10, int(Z * Z_DELTA_FRAC))

    fracs = _linspace_offsets(NUM_Z_OFFSETS)
    z_list = [int(np.clip(z0 + f * delta, 0, Z - 1)) for f in fracs]
    y_delta = int(Y * Y_SHIFT_FRAC)
    x_delta = int(X * X_SHIFT_FRAC)
    y_list = [int(np.clip(Y // 2 + f * y_delta, 0, Y - 1)) for f in fracs]
    x_list = [int(np.clip(X // 2 + f * x_delta, 0, X - 1)) for f in fracs]

    out_paths: List[str] = []
    for i, (z, y_idx, x_idx) in enumerate(zip(z_list, y_list, x_list)):
        montage = make_montage(v_r, v_g, v_b, z=z, y=y_idx, x=x_idx)
        out_path = os.path.join(IMG_DIR, f"{pid}_z{i}.png")
        montage.save(out_path)
        out_paths.append(out_path)

    return out_paths


_PAT_ENDO = re.compile(
    r"(?i)\bpulpitis\b|\bpulp\s*necrosis\b|\bperiapical\b|\bapical\s+periodontitis\b|\bapical\b.{0,20}\blesion\b|\bchronic\s+apical\b|\bacute\s+apical\b|K04\."
)
_PAT_PERIO = re.compile(r"(?i)\bgingivitis\b|\bperiodontitis\b|\bperiodontal\b|\bbone\s+loss\b|K05\.")
_PAT_DEV_POS = re.compile(
    r"(?i)\bmalocclusion\b|\bcrooked\b|\bmalaligned\b|\bmisaligned\b|\bimpacted\b|\bimpaction\b|\bnot\s+erupt\b|\bfailed\s+to\s+erupt\b|\bbuccal\s+eruption\b|\bmalformed\b|\brotated\b|K07\.|K01\."
)
_PAT_CARIES = re.compile(r"(?i)\bcaries\b|\bcavit(?:ies|y)\b|\bdecay\b|\bcarious\b|K02\.")
_PAT_STRUCT = re.compile(r"(?i)\bmissing\b|\bfractur(?:ed|e)\b|\bdefect\b|\bdamaged\b|K08\.")
_PROC_ONLY = re.compile(
    r"(?i)\bfollow[- ]?up\b|\bcheck\b|\bpost[- ]?op\b|\bsuture\s+removal\b|\btemporary\s+seal\b|\broot\s+filling\b|\bafter\s+implantation\b"
)
_FDI_RE = re.compile(r"(?:tooth\s*)?[*#]?\s*([1-4][1-8])\b")


def extract_fdi_teeth(text: str) -> List[str]:
    t = safe_str(text)
    return sorted(set(_FDI_RE.findall(t)), key=lambda s: (s[0], s[1])) if t else []


def extract_labels(text: str) -> List[str]:
    found = []
    if _PAT_ENDO.search(text):
        found.append("ENDO")
    if _PAT_PERIO.search(text):
        found.append("PERIO")
    if _PAT_DEV_POS.search(text):
        found.append("DEV_POS")
    if _PAT_CARIES.search(text) or _PAT_STRUCT.search(text):
        found.append("CARIES_STRUCT")
    return found


def patient_labels_from_group(group: pd.DataFrame) -> Tuple[List[str], List[str]]:
    index_row: Optional[pd.Series] = None
    for _, row in group.iterrows():
        if safe_str(row.get("Main appeal", "")):
            index_row = row
            break
    if index_row is None:
        index_row = group.iloc[0]

    dx = safe_str(index_row.get("Diagnosis", ""))
    oc = safe_str(index_row.get("Oral Check", ""))
    text = (dx + " " + oc).strip()

    labels = extract_labels(text)
    fdi = extract_fdi_teeth(text)

    if not labels:
        all_text = " ".join(
            safe_str(r.get("Diagnosis", "")) + " " + safe_str(r.get("Oral Check", ""))
            for _, r in group.iterrows()
        )
        labels = extract_labels(all_text)
        fdi = fdi or extract_fdi_teeth(all_text)

    if not labels and _PROC_ONLY.search(text):
        return [], []

    labels = [l for l in LABEL_SPACE if l in labels]
    return labels, fdi


_LABEL_DESC = {
    "ENDO": "endodontic pathology (pulpitis or periapical lesion)",
    "PERIO": "periodontal disease (bone loss or gingivitis)",
    "DEV_POS": "developmental or positional anomaly (malocclusion, impaction)",
    "CARIES_STRUCT": "caries, structural defect, or missing tooth",
}

_SYSTEM = (
    "You are a specialist dental radiologist interpreting 3D CBCT montage images. "
    "Each montage is a 3×3 grid: rows = axial / coronal / sagittal planes; "
    "columns = center slice / metal-suppressed MIP / slab mean projection. "
    "Colour channels: Red = bone window, Green = soft-tissue window, "
    "Blue = dense enamel / cortical bone."
)


def _img_block(path: str) -> Dict[str, str]:
    return {"type": "image", "image": path}


def _txt_block(text: str) -> Dict[str, str]:
    return {"type": "text", "text": text}


def _context(row: pd.Series, fdi: List[str]) -> str:
    age = safe_str(row.get("Age", "N/A"))
    sex = safe_str(row.get("Sex", "N/A"))
    appeal = safe_str(row.get("Main appeal", "not recorded"))
    fdi_s = ", ".join(fdi) if fdi else "none referenced"
    return f"Patient: {age}-year-old {sex}. Chief complaint: {appeal}. FDI teeth referenced in clinical record: {fdi_s}."


def build_describe_record(
    pid: str,
    z_idx: int,
    img_path: str,
    row: pd.Series,
    labels: List[str],
    fdi: List[str],
    meta_base: Dict[str, Any],
) -> Dict[str, Any]:
    dx = safe_str(row.get("Diagnosis", ""))
    oc = safe_str(row.get("Oral Check", ""))

    answer_parts = []
    if oc:
        answer_parts.append(f"Oral examination: {oc}")
    if dx:
        answer_parts.append(f"Diagnosis: {dx}")
    if labels:
        answer_parts.append("Pathology present: " + "; ".join(_LABEL_DESC[l] for l in labels) + ".")
    else:
        answer_parts.append("No significant pathology identified.")
    if fdi:
        answer_parts.append(f"Affected teeth (FDI): {', '.join(fdi)}.")

    return {
        "id": f"{pid}_z{z_idx}_{TASK_DESCRIBE}",
        "task": TASK_DESCRIBE,
        "messages": [
            {
                "role": "user",
                "content": [
                    _img_block(img_path),
                    _txt_block(f"{_SYSTEM}\n\n{_context(row, fdi)}\n\nDescribe all dental and osseous findings visible in this CBCT montage."),
                ],
            },
            {"role": "assistant", "content": [_txt_block(" ".join(answer_parts) if answer_parts else "Findings not recorded.")]},
        ],
        "meta": {**meta_base, "z_idx": z_idx, "task": TASK_DESCRIBE, "supervision_source": "csv_text"},
    }


def build_classify_record(
    pid: str,
    z_idx: int,
    img_path: str,
    row: pd.Series,
    labels: List[str],
    fdi: List[str],
    meta_base: Dict[str, Any],
) -> Dict[str, Any]:
    label_list = ", ".join(LABEL_SPACE)
    return {
        "id": f"{pid}_z{z_idx}_{TASK_CLASSIFY}",
        "task": TASK_CLASSIFY,
        "labels": labels,
        "messages": [
            {
                "role": "user",
                "content": [
                    _img_block(img_path),
                    _txt_block(
                        f"{_SYSTEM}\n\n{_context(row, fdi)}\n\nClassify this CBCT scan using the label space: [{label_list}].\nReturn ONLY a JSON array of applicable labels ([] if none)."
                    ),
                ],
            },
            {"role": "assistant", "content": [_txt_block(json.dumps(labels, ensure_ascii=False))]},
        ],
        "meta": {**meta_base, "z_idx": z_idx, "task": TASK_CLASSIFY, "supervision_source": "image_grounded"},
    }


def build_tooth_record(
    pid: str,
    z_idx: int,
    img_path: str,
    row: pd.Series,
    labels: List[str],
    fdi: List[str],
    meta_base: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    if not fdi:
        return None

    tooth = fdi[0]
    source_text = " ".join([safe_str(row.get("Diagnosis", "")), safe_str(row.get("Oral Check", ""))]).strip()
    sentences = re.split(r"(?<=[.!?])\s+|\n+", source_text)
    hits = [s.strip() for s in sentences if s.strip() and re.search(rf"\b{re.escape(tooth)}\b", s)]
    if hits:
        answer = " ".join(hits)
    else:
        ctx = source_text[:300] if source_text else "not available"
        answer = f"Tooth {tooth} is referenced. Clinical context: {ctx}"

    return {
        "id": f"{pid}_z{z_idx}_{TASK_TOOTH}",
        "task": TASK_TOOTH,
        "messages": [
            {
                "role": "user",
                "content": [
                    _img_block(img_path),
                    _txt_block(f"{_SYSTEM}\n\n{_context(row, fdi)}\n\nDoes tooth {tooth} show any abnormality in this CBCT? Describe the finding specifically."),
                ],
            },
            {"role": "assistant", "content": [_txt_block(answer)]},
        ],
        "meta": {
            **meta_base,
            "z_idx": z_idx,
            "task": TASK_TOOTH,
            "fdi_focus": tooth,
            "supervision_source": "csv_text",
        },
    }


def build_treatment_record(
    pid: str,
    z_idx: int,
    img_path: str,
    row: pd.Series,
    labels: List[str],
    fdi: List[str],
    meta_base: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    tp = safe_str(row.get("Treatment plan", ""))
    ha = safe_str(row.get("Handle", ""))
    answer = f"{tp} Procedure: {ha}" if tp and ha else tp or ha
    if not answer:
        return None

    return {
        "id": f"{pid}_z{z_idx}_{TASK_TREATMENT}",
        "task": TASK_TREATMENT,
        "messages": [
            {
                "role": "user",
                "content": [
                    _img_block(img_path),
                    _txt_block(f"{_SYSTEM}\n\n{_context(row, fdi)}\n\nWhat treatment approach is appropriate for the identified pathology?"),
                ],
            },
            {"role": "assistant", "content": [_txt_block(answer)]},
        ],
        "meta": {**meta_base, "z_idx": z_idx, "task": TASK_TREATMENT, "supervision_source": "csv_text"},
    }


def build_history_record(
    pid: str,
    z_idx: int,
    img_path: str,
    row: pd.Series,
    labels: List[str],
    fdi: List[str],
    meta_base: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    pmh = safe_str(row.get("Present medical history", ""))
    if not pmh:
        return None

    return {
        "id": f"{pid}_z{z_idx}_{TASK_HISTORY}",
        "task": TASK_HISTORY,
        "messages": [
            {
                "role": "user",
                "content": [
                    _img_block(img_path),
                    _txt_block(
                        f"{_SYSTEM}\n\n{_context(row, fdi)}\n\nAre there relevant medical history factors that could affect diagnosis or treatment planning for this patient?"
                    ),
                ],
            },
            {"role": "assistant", "content": [_txt_block(pmh)]},
        ],
        "meta": {**meta_base, "z_idx": z_idx, "task": TASK_HISTORY, "supervision_source": "csv_text"},
    }


def build_records_for_patient(pid: str, image_paths: List[str], index_row: pd.Series, labels: List[str], fdi: List[str]) -> List[Dict[str, Any]]:
    meta_base: Dict[str, Any] = {
        "patient_id": pid,
        "age": safe_str(index_row.get("Age", "")),
        "sex": safe_str(index_row.get("Sex", "")),
        "main_appeal": safe_str(index_row.get("Main appeal", "")),
        "fdi_teeth": fdi,
        "label_space": LABEL_SPACE,
        "pathology_labels": labels,
        "n_z_offsets": len(image_paths),
        "qwen_resolution": {"min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
    }

    records: List[Dict[str, Any]] = []
    builders = {
        TASK_DESCRIBE: build_describe_record,
        TASK_CLASSIFY: build_classify_record,
        TASK_TOOTH: build_tooth_record,
        TASK_TREATMENT: build_treatment_record,
        TASK_HISTORY: build_history_record,
    }

    for z_idx, img_path in enumerate(image_paths):
        for task in ENABLED_TASKS:
            rec = builders[task](pid, z_idx, img_path, index_row, labels, fdi, meta_base)
            if rec is not None:
                records.append(rec)

    return records


def compute_class_weights(label_counts: Dict[str, int]) -> Dict[str, float]:
    total = float(sum(label_counts.values()))
    n_cls = float(len(label_counts))
    raw = {cls: total / (n_cls * max(1, cnt)) for cls, cnt in label_counts.items()}
    mean_w = float(np.mean(list(raw.values())))
    return {cls: round(w / mean_w, 4) for cls, w in raw.items()}


def _primary_label(labels: List[str]) -> str:
    for l in ["ENDO", "PERIO", "DEV_POS", "CARIES_STRUCT"]:
        if l in labels:
            return l
    return "NORMAL"


def compute_sampler_weights(records: List[Dict[str, Any]]) -> List[float]:
    counts: Dict[str, int] = collections.Counter(
        _primary_label(r.get("meta", {}).get("pathology_labels", r.get("labels", []))) for r in records
    )
    return [
        1.0 / max(1, counts[_primary_label(r.get("meta", {}).get("pathology_labels", r.get("labels", [])))])
        for r in records
    ]


def print_class_report(split_records: Dict[str, List[Dict[str, Any]]]) -> Dict[str, Any]:
    print("\n  Class report  (label incidence — one record may carry multiple labels)")
    print(f"  {'Label':<20} {'Train':>8} {'Val':>8} {'Test':>8}  {'Tr %':>7} {'Va %':>7} {'Te %':>7}")
    print(f"  {'-' * 68}")

    report: Dict[str, Any] = {}
    totals = {sp: len(recs) for sp, recs in split_records.items()}

    for lbl in LABEL_SPACE + ["NORMAL"]:
        row_counts: Dict[str, int] = {}
        for sp, recs in split_records.items():
            if lbl == "NORMAL":
                row_counts[sp] = sum(1 for r in recs if not r.get("meta", {}).get("pathology_labels", r.get("labels", [])))
            else:
                row_counts[sp] = sum(1 for r in recs if lbl in r.get("meta", {}).get("pathology_labels", r.get("labels", [])))

        def _pct(sp: str) -> float:
            return row_counts[sp] / max(1, totals[sp]) * 100

        print(
            f"  {lbl:<20} {row_counts.get('train', 0):>8d} {row_counts.get('val', 0):>8d} {row_counts.get('test', 0):>8d}  {_pct('train'):>6.1f}% {_pct('val'):>6.1f}% {_pct('test'):>6.1f}%"
        )
        report[lbl] = row_counts

    print(f"  {'Total records':<20} {totals.get('train', 0):>8d} {totals.get('val', 0):>8d} {totals.get('test', 0):>8d}")
    return report


def stratified_split(
    patient_record_map: Dict[str, List[Dict[str, Any]]],
    seed: int = SEED,
    train: float = TRAIN_RATIO,
    val: float = VAL_RATIO,
    test: float = TEST_RATIO,
) -> Tuple[List[str], List[str], List[str]]:
    rng = random.Random(seed)
    buckets: Dict[str, List[str]] = collections.defaultdict(list)

    for pid, recs in patient_record_map.items():
        all_labels = list({l for r in recs for l in r.get("meta", {}).get("pathology_labels", r.get("labels", []))})
        buckets[_primary_label(all_labels)].append(pid)

    train_ids: List[str] = []
    val_ids: List[str] = []
    test_ids: List[str] = []

    for lbl in sorted(buckets):
        pids = list(buckets[lbl])
        rng.shuffle(pids)
        n = len(pids)

        if n < 3:
            train_ids.extend(pids)
            print(f"  [WARN] class '{lbl}' has only {n} patient(s) — all assigned to train; no val/test examples for this class.")
            continue

        n_val = max(1, int(round(n * val)))
        n_test = max(1, int(round(n * test)))
        if n - n_val - n_test < 1:
            n_val = max(1, n_val - 1)
        n_train = n - n_val - n_test

        train_ids.extend(pids[:n_train])
        val_ids.extend(pids[n_train:n_train + n_val])
        test_ids.extend(pids[n_train + n_val:])

    rng.shuffle(train_ids)
    rng.shuffle(val_ids)
    rng.shuffle(test_ids)
    return train_ids, val_ids, test_ids


def check_leakage(t: List[str], v: List[str], te: List[str]) -> bool:
    ts, vs, tes = set(t), set(v), set(te)
    bad = (ts & vs) | (ts & tes) | (vs & tes)
    if bad:
        print(f"  [ERROR] Patient leakage detected: {sorted(bad)[:10]} …")
        return False
    print("  [OK] No patient leakage across splits.")
    return True


def flatten(patient_record_map: Dict[str, List[Dict[str, Any]]], ids: List[str]) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for pid in ids:
        out.extend(patient_record_map.get(pid, []))
    return out


def save_splits(
    patient_record_map: Dict[str, List[Dict[str, Any]]],
    train_ids: List[str],
    val_ids: List[str],
    test_ids: List[str],
) -> Dict[str, str]:
    paths: Dict[str, str] = {}
    for name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
        recs = flatten(patient_record_map, ids)
        p = os.path.join(SPLIT_DIR, f"{name}.jsonl")
        write_jsonl(p, recs)
        paths[name] = p

        if name == "train":
            sw_path = os.path.join(SPLIT_DIR, "train_sampler_weights.json")
            with open(sw_path, "w", encoding="utf-8") as f:
                json.dump({"weights": compute_sampler_weights(recs), "n_records": len(recs)}, f, indent=2)
            paths["sampler_weights"] = sw_path

    return paths


def main() -> None:
    print("=" * 60)
    print("MMDental Preprocessing Pipeline")
    print("=" * 60)

    print("\n[1/6] Loading CSV …")
    df = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip().str.lstrip("\ufeff")
    df["Filename"] = df["Filename"].apply(lambda x: _canon_id(x))

    required = [
        "Filename",
        "Sex",
        "Age",
        "Main appeal",
        "Oral Check",
        "Diagnosis",
        "Treatment plan",
        "Handle",
        "Present medical history",
    ]
    for col in required:
        if col not in df.columns:
            df[col] = ""

    print(f"  Rows: {len(df)}  |  Patients: {df['Filename'].nunique()}")

    print("\n[2/6] Indexing volumes …")
    vol_index = build_volume_index(DATASET_PATH)
    print(f"  Indexed {len(vol_index)} volume keys")

    print("\n[3/6] Processing patients …")
    print(f"  NOTE: Debug arch slices → {DEBUG_DIR}")
    print("        Inspect these on first run to verify z-detection.\n")

    grouped = df.groupby("Filename", sort=False)
    patient_ids = list(grouped.groups.keys())

    patient_record_map: Dict[str, List[Dict[str, Any]]] = {}
    n_missing, n_skipped, n_ok = 0, 0, 0
    missing_log: List[str] = []

    for pid in tqdm(patient_ids, desc="  Patients"):
        found = find_volume(pid, vol_index)
        if found is None:
            n_missing += 1
            missing_log.append(pid)
            continue

        kind, vol_path = found
        group = grouped.get_group(pid)

        labels, fdi = patient_labels_from_group(group)
        if not KEEP_NEGATIVES and not labels:
            n_skipped += 1
            continue

        expected = [os.path.join(IMG_DIR, f"{pid}_z{i}.png") for i in range(NUM_Z_OFFSETS)]
        if all(os.path.exists(p) for p in expected):
            image_paths = expected
        else:
            image_paths = create_patient_images(pid, kind, vol_path)
            if image_paths is None:
                n_missing += 1
                missing_log.append(pid)
                continue

        index_row: pd.Series = next((row for _, row in group.iterrows() if safe_str(row.get("Main appeal", ""))), group.iloc[0])
        records = build_records_for_patient(pid, image_paths, index_row, labels, fdi)
        patient_record_map[pid] = records
        n_ok += 1

    print(f"\n  OK: {n_ok}  |  Missing volume: {n_missing}  |  Skipped: {n_skipped}")

    if not patient_record_map:
        print("[ERROR] No records produced. Check paths and volume index.")
        return

    all_records = [r for recs in patient_record_map.values() for r in recs]
    write_jsonl(JSONL_SOURCE, all_records)
    print(f"\n[4/6] Source JSONL: {JSONL_SOURCE}")
    print(f"  Total records: {len(all_records)}  ({len(patient_record_map)} patients × ~{len(all_records) // max(1, len(patient_record_map))} records each)")

    print("\n[5/6] Splitting …")
    train_ids, val_ids, test_ids = stratified_split(patient_record_map)
    print(f"  Train patients : {len(train_ids)}")
    print(f"  Val   patients : {len(val_ids)}")
    print(f"  Test  patients : {len(test_ids)}")
    check_leakage(train_ids, val_ids, test_ids)
    split_paths = save_splits(patient_record_map, train_ids, val_ids, test_ids)
    print(f"  Splits → {SPLIT_DIR}")

    print("\n[6/6] Class distribution & imbalance report …")
    split_rec_map = {
        "train": flatten(patient_record_map, train_ids),
        "val": flatten(patient_record_map, val_ids),
        "test": flatten(patient_record_map, test_ids),
    }
    dist = print_class_report(split_rec_map)

    train_label_counts: Dict[str, int] = {lbl: 0 for lbl in LABEL_SPACE + ["NORMAL"]}
    for r in split_rec_map["train"]:
        pathology_labels = r.get("meta", {}).get("pathology_labels", r.get("labels", []))
        for l in (pathology_labels or ["NORMAL"]):
            train_label_counts[l] += 1
    cw = compute_class_weights(train_label_counts)
    print("\n  Class weights (train-only, for loss function):")
    for cls, w in cw.items():
        print(f"    {cls:<20}  weight = {w:.4f}")

    checksums = {
        "source_csv_sha256": sha256(CSV_PATH),
        "source_jsonl_sha256": sha256(JSONL_SOURCE),
        "train_jsonl_sha256": sha256(split_paths["train"]),
        "val_jsonl_sha256": sha256(split_paths["val"]),
        "test_jsonl_sha256": sha256(split_paths["test"]),
        "config": {
            "CRANIAL_FRAC": CRANIAL_FRAC,
            "NUM_Z_OFFSETS": NUM_Z_OFFSETS,
            "ISOTROPIC_SPACING": list(ISOTROPIC_SPACING),
            "SEED": SEED,
            "TRAIN_RATIO": TRAIN_RATIO,
            "VAL_RATIO": VAL_RATIO,
            "TEST_RATIO": TEST_RATIO,
            "LABEL_SPACE": LABEL_SPACE,
            "ENABLED_TASKS": ENABLED_TASKS,
            "KEEP_NEGATIVES": KEEP_NEGATIVES,
        },
    }

    stats = {
        "pipeline_version": "v3",
        "cranial_frac_applied": CRANIAL_FRAC,
        "n_patients_total": len(patient_ids),
        "n_patients_ok": n_ok,
        "n_patients_missing": n_missing,
        "n_patients_skipped": n_skipped,
        "n_records_total": len(all_records),
        "split_patient_counts": {"train": len(train_ids), "val": len(val_ids), "test": len(test_ids)},
        "split_record_counts": {sp: len(recs) for sp, recs in split_rec_map.items()},
        "label_incidence_per_split": dist,
        "class_weights_train": cw,
        "paths": split_paths,
        "missing_volume_ids": missing_log,
        "checksums": checksums,
    }

    with open(STATS_OUT, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(stats), f, indent=2, ensure_ascii=False)

    print(f"\n[DONE] Stats: {STATS_OUT}")
    print("=" * 60)


if __name__ == "__main__":
    main()


MMDental Preprocessing Pipeline

[1/6] Loading CSV …
  Rows: 2124  |  Patients: 660

[2/6] Indexing volumes …
  Indexed 1672 volume keys

[3/6] Processing patients …
  NOTE: Debug arch slices → /content/drive/MyDrive/Dental_Project/MMDental_Preprocessed/debug_arch_slices
        Inspect these on first run to verify z-detection.



  Patients: 100%|██████████| 660/660 [1:54:54<00:00, 10.45s/it]



  OK: 403  |  Missing volume: 257  |  Skipped: 0

[4/6] Source JSONL: /content/drive/MyDrive/Dental_Project/MMDental_Preprocessed/source.jsonl
  Total records: 5457  (403 patients × ~13 records each)

[5/6] Splitting …
  Train patients : 283
  Val   patients : 60
  Test  patients : 60
  [OK] No patient leakage across splits.
  Splits → /content/drive/MyDrive/Dental_Project/MMDental_Preprocessed/splits

[6/6] Class distribution & imbalance report …

  Class report  (label incidence — one record may carry multiple labels)
  Label                   Train      Val     Test     Tr %    Va %    Te %
  --------------------------------------------------------------------
  ENDO                     1140      249      243    29.8%   29.9%   30.6%
  PERIO                    1026      237      285    26.8%   28.4%   35.8%
  DEV_POS                  1173      201      216    30.6%   24.1%   27.2%
  CARIES_STRUCT            2592      567      528    67.7%   68.0%   66.4%
  NORMAL                   

In [ ]:
import os

# 1. Path to your main folder
folder_path = '/content/drive/MyDrive/Dental_Project/MMDental_Preprocessed'

# 2. Check if the directory exists
if os.path.exists(folder_path):
    # List all items in the root directory
    items = os.listdir(folder_path)
    items.sort()

    print(f"Successfully accessed: {folder_path}")
    print(f"Total items in root: {len(items)}")
    print("-" * 40)

    for item in items:
        item_path = os.path.join(folder_path, item)

        # Check if the item is a folder
        if os.path.isdir(item_path):
            sub_files = os.listdir(item_path)
            sub_files.sort()

            print(f"\n[FOLDER] {item}")
            print(f"   - Total files inside: {len(sub_files)}")

            # Print the first 5 samples from this subfolder
            for sub_file in sub_files[:5]:
                print(f"   --> {sub_file}")

            if len(sub_files) > 5:
                print(f"   ... and {len(sub_files) - 5} more files in this folder.")

        else:
            # It's a file in the root directory
            print(f"[FILE]   {item}")

else:
    print(f"Error: The folder '{folder_path}' does not exist.")
    print("Ensure your Google Drive is mounted correctly.")

Successfully accessed: /content/drive/MyDrive/Dental_Project/MMDental_Preprocessed
Total items in root: 5
----------------------------------------
[FILE]   dataset_stats.json

[FOLDER] debug_arch_slices
   - Total files inside: 806
   --> 100_arch_z084.png
   --> 100_crop_arch_z084.png
   --> 102_arch_z120.png
   --> 102_crop_arch_z120.png
   --> 104_arch_z126.png
   ... and 801 more files in this folder.

[FOLDER] images
   - Total files inside: 1209
   --> 100_z0.png
   --> 100_z1.png
   --> 100_z2.png
   --> 102_z0.png
   --> 102_z1.png
   ... and 1204 more files in this folder.
[FILE]   source.jsonl

[FOLDER] splits
   - Total files inside: 4
   --> test.jsonl
   --> train.jsonl
   --> train_sampler_weights.json
   --> val.jsonl


# **DATASET PRE-PROCESSING (ToothFairy Dataset)**


In [ ]:
import os

# 1. Path to your main folder
folder_path = '/content/drive/MyDrive/Dental_Project/ToothFairy2'

# 2. Check if the directory exists
if os.path.exists(folder_path):
    # List all items in the root directory
    items = os.listdir(folder_path)
    items.sort()

    print(f"Successfully accessed: {folder_path}")
    print(f"Total items in root: {len(items)}")
    print("-" * 40)

    for item in items:
        item_path = os.path.join(folder_path, item)

        # Check if the item is a folder
        if os.path.isdir(item_path):
            sub_files = os.listdir(item_path)
            sub_files.sort()

            print(f"\n[FOLDER] {item}")
            print(f"   - Total files inside: {len(sub_files)}")

            # Print the first 5 samples from this subfolder
            for sub_file in sub_files[:5]:
                print(f"   --> {sub_file}")

            if len(sub_files) > 5:
                print(f"   ... and {len(sub_files) - 5} more files in this folder.")

        else:
            # It's a file in the root directory
            print(f"[FILE]   {item}")

else:
    print(f"Error: The folder '{folder_path}' does not exist.")
    print("Ensure your Google Drive is mounted correctly.")

Successfully accessed: /content/drive/MyDrive/Dental_Project/ToothFairy2
Total items in root: 3
----------------------------------------
[FILE]   dataset.json

[FOLDER] imagesTr
   - Total files inside: 480
   --> ToothFairy2F_001_0000.mha
   --> ToothFairy2F_002_0000.mha
   --> ToothFairy2F_003_0000.mha
   --> ToothFairy2F_004_0000.mha
   --> ToothFairy2F_005_0000.mha
   ... and 475 more files in this folder.

[FOLDER] labelsTr
   - Total files inside: 480
   --> ToothFairy2F_001.mha
   --> ToothFairy2F_002.mha
   --> ToothFairy2F_003.mha
   --> ToothFairy2F_004.mha
   --> ToothFairy2F_005.mha
   ... and 475 more files in this folder.


In [ ]:
# -*- coding: utf-8 -*-
"""
ToothFairy2 Preprocessing Pipeline  —  v4 (reconstruction-ready)
================================================================
This version preserves the original VQA-style outputs and additionally exports
montage-aligned dense reconstruction targets for Stage 1 modelling.

What is fixed vs the previous version:
  - Debug overlays remain QA visualisations only and are no longer treated as
    recoverable supervision targets.
  - For every saved montage image, a true dense 3-channel reconstruction target
    is exported directly from vol_lbl using the same z / y / x / slab geometry
    that generated the montage.
  - The dense target is montage-aligned: rows = axial / coronal / sagittal,
    columns = center slice / MIP / slab mean, matching the 3×3 image layout.
  - Each TF2 JSONL record now carries recon_target_npy and recon_target_png paths
    in meta so training can load exact targets without reverse-engineering
    blended overlay PNGs.

Label map
---------
  1-16  : upper teeth (FDI Q1 + Q2)
  17-32 : lower teeth (FDI Q3 + Q4)
  33-40 : jaw bones (mandible, maxilla, processes)
  41    : left inferior alveolar canal (IAC)
  42    : right inferior alveolar canal (IAC)
  0     : background

Dense target channels
---------------------
  channel 0 : Teeth (upper + lower)
  channel 1 : Jaws
  channel 2 : IAC (left + right)

Exported directories
--------------------
  images/         : RGB 3×3 CBCT montages
  debug_overlays/ : axial QA overlays only
  recon_targets/  : exact montage-aligned (3, H, W) float32 targets as .npy
  recon_pngs/     : RGB visualisation of the dense targets
  splits/         : train / val / test JSONL + sampler weights
"""

import collections
import glob
import hashlib
import json
import os
import random
import re
import warnings
from typing import Any, Dict, List, Optional, Set, Tuple

import cv2
import numpy as np
import PIL.Image
import SimpleITK as sitk
import tqdm

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────
# 1.  CONFIG
# ─────────────────────────────────────────────────────────
TF2_ROOT   = "/content/drive/MyDrive/Dental_Project/ToothFairy2"
IMAGES_DIR = os.path.join(TF2_ROOT, "imagesTr")
LABELS_DIR = os.path.join(TF2_ROOT, "labelsTr")

OUT_DIR    = "/content/drive/MyDrive/Dental_Project/ToothFairy2_Preprocessed"
IMG_DIR    = os.path.join(OUT_DIR, "images")
DBG_DIR    = os.path.join(OUT_DIR, "debug_overlays")
RECON_DIR  = os.path.join(OUT_DIR, "recon_targets")
RECON_PNG  = os.path.join(OUT_DIR, "recon_pngs")
SPLIT_DIR  = os.path.join(OUT_DIR, "splits")
STATS_OUT  = os.path.join(OUT_DIR, "dataset_stats.json")
JSONL_SRC  = os.path.join(OUT_DIR, "source.jsonl")

for _d in [OUT_DIR, IMG_DIR, DBG_DIR, RECON_DIR, RECON_PNG, SPLIT_DIR]:
    os.makedirs(_d, exist_ok=True)

TARGET_SPACING: Tuple[float, float, float] = (0.3, 0.3, 0.3)

TILE_HW       = (448, 448)
NUM_Z_OFFSETS = 3
SLAB_FRAC     = 0.07
MIP_CAP_P     = 99.0
SAVE_DEBUG    = True

UPPER_TEETH  = list(range(1, 17))
LOWER_TEETH  = list(range(17, 33))
TEETH_LABELS = UPPER_TEETH + LOWER_TEETH
JAW_LABELS   = list(range(33, 41))
IAC_LEFT     = 41
IAC_RIGHT    = 42
IAC_LABELS   = [IAC_LEFT, IAC_RIGHT]

LABEL_ORDER = ["Teeth", "Jaws", "IAC"]
N_RECON_CH  = 3
RECON_HW    = 128

MIN_VISIBLE_VOXELS = 300

TASK_CLASSIFY = "anatomy_classify"
TASK_DESCRIBE = "anatomy_describe"
TASK_ARCH     = "tooth_arch"
TASK_IAC      = "iac_presence"
TASK_COUNT    = "structure_count"
ENABLED_TASKS = [TASK_CLASSIFY, TASK_DESCRIBE, TASK_ARCH, TASK_IAC, TASK_COUNT]

_SRC_MASK      = "mask_grounded"
_SRC_MASK_DESC = "mask_derived"
_SRC_RECON     = "mask_reconstruction_target"

TASK_SRC = {
    TASK_CLASSIFY: _SRC_MASK,
    TASK_DESCRIBE: _SRC_MASK_DESC,
    TASK_ARCH:     _SRC_MASK,
    TASK_IAC:      _SRC_MASK,
    TASK_COUNT:    _SRC_MASK,
}

SEED        = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

MIN_PIXELS = 256 * 256
MAX_PIXELS = 1344 * 1344

CONFIG_SNAPSHOT: Dict[str, Any] = {
    "pipeline":               "toothfairy2_v4_reconstruction_ready",
    "TARGET_SPACING":         list(TARGET_SPACING),
    "TILE_HW":                list(TILE_HW),
    "NUM_Z_OFFSETS":          NUM_Z_OFFSETS,
    "SLAB_FRAC":              SLAB_FRAC,
    "MIP_CAP_P":              MIP_CAP_P,
    "MIN_VISIBLE_VOXELS":     MIN_VISIBLE_VOXELS,
    "MIN_VISIBLE_NOTE":       "threshold is sum across axial+coronal+sagittal, not per-axis",
    "LABEL_ORDER":            LABEL_ORDER,
    "N_RECON_CH":             N_RECON_CH,
    "RECON_HW":               RECON_HW,
    "ENABLED_TASKS":          ENABLED_TASKS,
    "SEED":                   SEED,
    "TRAIN_RATIO":            TRAIN_RATIO,
    "VAL_RATIO":              VAL_RATIO,
    "TEST_RATIO":             TEST_RATIO,
}


# ─────────────────────────────────────────────────────────
# 2.  UTILITIES
# ─────────────────────────────────────────────────────────
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def write_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(to_jsonable(r), ensure_ascii=False) + "\n")


def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ─────────────────────────────────────────────────────────
# 3.  DATASET INDEXING
# ─────────────────────────────────────────────────────────
_ID_RE = re.compile(r"(ToothFairy2[FP]_\d+)")


def index_dataset(images_dir: str, labels_dir: str) -> Dict[str, Dict[str, str]]:
    def _scan(d: str) -> Dict[str, str]:
        idx: Dict[str, str] = {}
        for p in glob.glob(os.path.join(d, "*.mha")):
            m = _ID_RE.search(os.path.basename(p))
            if m:
                idx[m.group(1)] = p
        return idx

    img_idx = _scan(images_dir)
    lbl_idx = _scan(labels_dir)
    matched = {cid: {"image": img_idx[cid], "label": lbl_idx[cid]} for cid in sorted(img_idx.keys() & lbl_idx.keys())}

    miss_lbl = sorted(img_idx.keys() - lbl_idx.keys())
    miss_img = sorted(lbl_idx.keys() - img_idx.keys())
    if miss_lbl:
        print(f"  [WARN] {len(miss_lbl)} images without labels: {miss_lbl[:5]}")
    if miss_img:
        print(f"  [WARN] {len(miss_img)} labels without images: {miss_img[:5]}")
    print(f"  Matched pairs: {len(matched)}")
    return matched


# ─────────────────────────────────────────────────────────
# 4.  VOLUME LOADING & RESAMPLING
# ─────────────────────────────────────────────────────────
def _load_orient(path: str) -> sitk.Image:
    img = sitk.ReadImage(path)
    try:
        img = sitk.DICOMOrient(img, "LPS")
    except Exception:
        pass
    return img


def resample_image(img: sitk.Image, spacing: Tuple[float, ...] = TARGET_SPACING) -> sitk.Image:
    sp  = img.GetSpacing()
    sz  = img.GetSize()
    out = [max(1, int(round(sz[i] * sp[i] / spacing[i]))) for i in range(3)]
    rs  = sitk.ResampleImageFilter()
    rs.SetOutputSpacing(spacing)
    rs.SetSize(out)
    rs.SetOutputDirection(img.GetDirection())
    rs.SetOutputOrigin(img.GetOrigin())
    rs.SetInterpolator(sitk.sitkLinear)
    rs.SetDefaultPixelValue(0)
    return rs.Execute(img)


def resample_label_to_reference(label: sitk.Image, ref: sitk.Image) -> sitk.Image:
    rs = sitk.ResampleImageFilter()
    rs.SetReferenceImage(ref)
    rs.SetInterpolator(sitk.sitkNearestNeighbor)
    rs.SetDefaultPixelValue(0)
    return rs.Execute(label)


def load_case(image_path: str, label_path: str) -> Tuple[np.ndarray, np.ndarray]:
    img_sitk = _load_orient(image_path)
    lbl_sitk = _load_orient(label_path)
    img_rs   = resample_image(img_sitk, TARGET_SPACING)
    lbl_rs   = resample_label_to_reference(lbl_sitk, img_rs)
    vol_img  = sitk.GetArrayFromImage(img_rs).astype(np.float32)
    vol_lbl  = sitk.GetArrayFromImage(lbl_rs).astype(np.uint8)
    if vol_img.shape != vol_lbl.shape:
        raise ValueError(f"Shape mismatch: image={vol_img.shape} label={vol_lbl.shape}")
    return vol_img, vol_lbl


# ─────────────────────────────────────────────────────────
# 5.  NORMALISATION
# ─────────────────────────────────────────────────────────
def is_hu_like(vol: np.ndarray) -> bool:
    p1, p99 = float(np.nanpercentile(vol, 1)), float(np.nanpercentile(vol, 99))
    vmin, vmax = float(np.nanmin(vol)), float(np.nanmax(vol))
    return bool(
        ((p1 < -200) and (p99 > 800))
        or ((vmin < -500) and (vmax > 1500))
        or ((p99 - p1) > 1000)
        or ((vmax - vmin) > 2000 and vmax > 1000)
    )


def window_norm(vol: np.ndarray, level: float, width: float) -> np.ndarray:
    lo, hi = level - width / 2.0, level + width / 2.0
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0.0, 1.0)


def robust_norm(vol: np.ndarray, p_lo: float = 1.0, p_hi: float = 99.0) -> np.ndarray:
    lo = float(np.nanpercentile(vol, p_lo))
    hi = float(np.nanpercentile(vol, p_hi))
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0.0, 1.0)


def build_rgb_channels(vol: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if is_hu_like(vol):
        return (
            window_norm(vol, 400.0, 1500.0),
            window_norm(vol, 40.0, 400.0),
            window_norm(vol, 1500.0, 2000.0),
        )
    return (
        robust_norm(vol, 1.0, 97.0),
        robust_norm(vol, 20.0, 80.0),
        robust_norm(vol, 75.0, 99.9),
    )


# ─────────────────────────────────────────────────────────
# 6.  MASK ANALYSIS
# ─────────────────────────────────────────────────────────
def _slab_bounds(length: int, idx: int, slab: int) -> Tuple[int, int]:
    return max(0, idx - slab), min(length, idx + slab + 1)


def _slab_sum(vol: np.ndarray, idx: int, slab: int, axis: int) -> float:
    s, e = _slab_bounds(vol.shape[axis], idx, slab)
    return float(np.take(vol, np.arange(s, e), axis=axis).sum())


def _slab_present(data: np.ndarray, idx: int, slab: int, axis: int) -> Set[int]:
    s, e = _slab_bounds(data.shape[axis], idx, slab)
    return set(np.unique(np.take(data, np.arange(s, e), axis=axis)).tolist())


def montage_semantics(vol_lbl: np.ndarray, z: int, y: int, x: int, slab_frac: float = SLAB_FRAC, threshold: int = MIN_VISIBLE_VOXELS) -> Dict[str, Any]:
    Z, Y, X = vol_lbl.shape
    sz = max(1, int(Z * slab_frac))
    sy = max(1, int(Y * slab_frac))
    sx = max(1, int(X * slab_frac))

    v_teeth = np.isin(vol_lbl, TEETH_LABELS).astype(np.float32)
    v_jaws  = np.isin(vol_lbl, JAW_LABELS).astype(np.float32)
    v_iac   = np.isin(vol_lbl, IAC_LABELS).astype(np.float32)

    scores: Dict[str, float] = {
        "Teeth": (_slab_sum(v_teeth, z, sz, 0) + _slab_sum(v_teeth, y, sy, 1) + _slab_sum(v_teeth, x, sx, 2)),
        "Jaws":  (_slab_sum(v_jaws,  z, sz, 0) + _slab_sum(v_jaws,  y, sy, 1) + _slab_sum(v_jaws,  x, sx, 2)),
        "IAC":   (_slab_sum(v_iac,   z, sz, 0) + _slab_sum(v_iac,   y, sy, 1) + _slab_sum(v_iac,   x, sx, 2)),
    }
    visible = [lbl for lbl in LABEL_ORDER if scores[lbl] >= threshold]

    present_ids: Set[int] = set()
    for ax, idx, slab in [(0, z, sz), (1, y, sy), (2, x, sx)]:
        present_ids |= _slab_present(vol_lbl, idx, slab, ax)

    teeth_visible = "Teeth" in visible
    iac_visible   = "IAC" in visible

    upper_ids = sorted(present_ids & set(UPPER_TEETH)) if teeth_visible else []
    lower_ids = sorted(present_ids & set(LOWER_TEETH)) if teeth_visible else []
    iac_left  = (IAC_LEFT in present_ids) and iac_visible
    iac_right = (IAC_RIGHT in present_ids) and iac_visible

    return {
        "visible_labels": visible,
        "n_upper": len(upper_ids),
        "n_lower": len(lower_ids),
        "upper_teeth_ids": upper_ids,
        "lower_teeth_ids": lower_ids,
        "n_total_teeth": len(upper_ids) + len(lower_ids),
        "iac_left": iac_left,
        "iac_right": iac_right,
        "present_ids": sorted(present_ids),
    }


def slab_centroid_yx(vol_lbl: np.ndarray, z: int, slab: int) -> Tuple[int, int]:
    Z, Y, X = vol_lbl.shape
    z1, z2 = _slab_bounds(Z, z, slab)
    fg = np.argwhere(vol_lbl[z1:z2] > 0)
    if len(fg) == 0:
        return Y // 2, X // 2
    return int(np.median(fg[:, 1])), int(np.median(fg[:, 2]))


def sample_z_positions(vol_lbl: np.ndarray, n: int = NUM_Z_OFFSETS) -> List[int]:
    z_fg = np.where(np.any(vol_lbl > 0, axis=(1, 2)))[0]
    if len(z_fg) == 0:
        z_fg = np.arange(vol_lbl.shape[0])
    z_min, z_max = int(z_fg[0]), int(z_fg[-1])
    if n == 1:
        return [(z_min + z_max) // 2]
    return [int(round(v)) for v in np.linspace(z_min, z_max, n)]


# ─────────────────────────────────────────────────────────
# 7.  VIEW EXTRACTION & MONTAGE
# ─────────────────────────────────────────────────────────
def _capped_mip(stack: np.ndarray, axis: int = 0) -> np.ndarray:
    cap = float(np.nanpercentile(stack, MIP_CAP_P))
    return np.max(np.minimum(stack, cap), axis=axis)


def _aspect_resize(img: np.ndarray, out_hw: Tuple[int, int]) -> np.ndarray:
    oh, ow = out_hw
    h, w = img.shape
    scale = min(ow / max(w, 1), oh / max(h, 1))
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))
    u8 = np.clip(img * 255, 0, 255).astype(np.uint8)
    u8 = cv2.resize(u8, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((oh, ow), dtype=np.uint8)
    canvas[(oh - nh) // 2:(oh - nh) // 2 + nh, (ow - nw) // 2:(ow - nw) // 2 + nw] = u8
    return canvas.astype(np.float32) / 255.0


def extract_views(v: np.ndarray, z: int, y: int, x: int) -> Dict[str, np.ndarray]:
    Z, Y, X = v.shape
    z = int(np.clip(z, 0, Z - 1))
    y = int(np.clip(y, 0, Y - 1))
    x = int(np.clip(x, 0, X - 1))
    sz = max(12, int(Z * SLAB_FRAC))
    sy = max(12, int(Y * SLAB_FRAC))
    sx = max(12, int(X * SLAB_FRAC))

    def _p(data, idx, slab, axis):
        s, e = max(0, idx - slab), min(data.shape[axis], idx + slab + 1)
        sl = np.take(data, np.arange(s, e), axis=axis)
        return (
            np.take(data, idx, axis=axis),
            _capped_mip(sl, axis=axis),
            np.mean(sl, axis=axis),
        )

    ax_s, ax_m, ax_mn = _p(v, z, sz, 0)
    cor_s, cor_m, cor_mn = _p(v, y, sy, 1)
    sag_s, sag_m, sag_mn = _p(v, x, sx, 2)
    cor_s, cor_m, cor_mn = [np.rot90(i, 1) for i in (cor_s, cor_m, cor_mn)]
    sag_s, sag_m, sag_mn = [np.rot90(i, 1) for i in (sag_s, sag_m, sag_mn)]

    return {
        "axial": ax_s,
        "axial_mip": ax_m,
        "axial_mean": ax_mn,
        "coronal": cor_s,
        "coronal_mip": cor_m,
        "coronal_mean": cor_mn,
        "sagittal": sag_s,
        "sagittal_mip": sag_m,
        "sagittal_mean": sag_mn,
    }


def make_montage(v_r: np.ndarray, v_g: np.ndarray, v_b: np.ndarray, z: int, y: int, x: int) -> PIL.Image.Image:
    row_keys = [
        ["axial", "axial_mip", "axial_mean"],
        ["coronal", "coronal_mip", "coronal_mean"],
        ["sagittal", "sagittal_mip", "sagittal_mean"],
    ]

    def _grid(views):
        return np.vstack([np.hstack([_aspect_resize(views[k], TILE_HW) for k in rk]) for rk in row_keys])

    vr, vg, vb = (extract_views(ch, z, y, x) for ch in (v_r, v_g, v_b))
    rgb = np.stack([np.clip(_grid(v) * 255, 0, 255).astype(np.uint8) for v in (vr, vg, vb)], axis=-1)
    return PIL.Image.fromarray(rgb, "RGB")


# ─────────────────────────────────────────────────────────
# 8.  DEBUG OVERLAY
# ─────────────────────────────────────────────────────────
_OVERLAY_COLOURS: Dict[str, Tuple[int, int, int]] = {
    "teeth_upper": (0, 255, 50),
    "teeth_lower": (0, 180, 255),
    "jaws":        (255, 80, 80),
    "iac":         (255, 0, 200),
}


def make_debug_overlay(vol_img: np.ndarray, vol_lbl: np.ndarray, z: int, alpha: float = 0.45) -> PIL.Image.Image:
    img_sl  = np.clip(robust_norm(vol_img[z]) * 255, 0, 255).astype(np.uint8)
    bgr     = cv2.cvtColor(img_sl, cv2.COLOR_GRAY2BGR)
    lbl_sl  = vol_lbl[z]
    overlay = bgr.copy()
    for ids, key in [(UPPER_TEETH, "teeth_upper"), (LOWER_TEETH, "teeth_lower"), (JAW_LABELS, "jaws"), (IAC_LABELS, "iac")]:
        mask = np.isin(lbl_sl, ids)
        if mask.any():
            overlay[mask] = _OVERLAY_COLOURS[key]
    result = cv2.addWeighted(bgr, 1 - alpha, overlay, alpha, 0)
    return PIL.Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))


# ─────────────────────────────────────────────────────────
# 9.  DENSE RECONSTRUCTION TARGET EXPORT
# ─────────────────────────────────────────────────────────
def _group_volume(vol_lbl: np.ndarray, label_ids: List[int]) -> np.ndarray:
    return np.isin(vol_lbl, label_ids).astype(np.float32)


def _resize_binary(img: np.ndarray, out_hw: Tuple[int, int]) -> np.ndarray:
    oh, ow = out_hw
    h, w = img.shape
    scale = min(ow / max(w, 1), oh / max(h, 1))
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))
    u8 = (img > 0.5).astype(np.uint8) * 255
    u8 = cv2.resize(u8, (nw, nh), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((oh, ow), dtype=np.uint8)
    canvas[(oh - nh) // 2:(oh - nh) // 2 + nh, (ow - nw) // 2:(ow - nw) // 2 + nw] = u8
    return (canvas > 127).astype(np.float32)


def extract_mask_views(v: np.ndarray, z: int, y: int, x: int) -> Dict[str, np.ndarray]:
    Z, Y, X = v.shape
    z = int(np.clip(z, 0, Z - 1))
    y = int(np.clip(y, 0, Y - 1))
    x = int(np.clip(x, 0, X - 1))
    sz = max(12, int(Z * SLAB_FRAC))
    sy = max(12, int(Y * SLAB_FRAC))
    sx = max(12, int(X * SLAB_FRAC))

    def _p(data, idx, slab, axis):
        s, e = max(0, idx - slab), min(data.shape[axis], idx + slab + 1)
        sl = np.take(data, np.arange(s, e), axis=axis)
        center = np.take(data, idx, axis=axis)
        mip = np.max(sl, axis=axis)
        mean = (np.mean(sl, axis=axis) > 0.0).astype(np.float32)
        return center.astype(np.float32), mip.astype(np.float32), mean.astype(np.float32)

    ax_s, ax_m, ax_mn = _p(v, z, sz, 0)
    cor_s, cor_m, cor_mn = _p(v, y, sy, 1)
    sag_s, sag_m, sag_mn = _p(v, x, sx, 2)
    cor_s, cor_m, cor_mn = [np.rot90(i, 1) for i in (cor_s, cor_m, cor_mn)]
    sag_s, sag_m, sag_mn = [np.rot90(i, 1) for i in (sag_s, sag_m, sag_mn)]

    return {
        "axial": ax_s,
        "axial_mip": ax_m,
        "axial_mean": ax_mn,
        "coronal": cor_s,
        "coronal_mip": cor_m,
        "coronal_mean": cor_mn,
        "sagittal": sag_s,
        "sagittal_mip": sag_m,
        "sagittal_mean": sag_mn,
    }


def make_recon_target(vol_lbl: np.ndarray, z: int, y: int, x: int, out_hw: int = RECON_HW) -> np.ndarray:
    row_keys = [
        ["axial", "axial_mip", "axial_mean"],
        ["coronal", "coronal_mip", "coronal_mean"],
        ["sagittal", "sagittal_mip", "sagittal_mean"],
    ]

    groups = [
        _group_volume(vol_lbl, TEETH_LABELS),
        _group_volume(vol_lbl, JAW_LABELS),
        _group_volume(vol_lbl, IAC_LABELS),
    ]

    chans: List[np.ndarray] = []
    for g in groups:
        views = extract_mask_views(g, z, y, x)
        grid = np.vstack([np.hstack([_aspect_resize(views[k], TILE_HW) for k in rk]) for rk in row_keys]).astype(np.float32)
        chans.append(_resize_binary(grid, (out_hw, out_hw)))
    return np.stack(chans, axis=0).astype(np.float32)


def make_recon_png(target: np.ndarray) -> PIL.Image.Image:
    rgb = np.zeros((target.shape[1], target.shape[2], 3), dtype=np.uint8)
    teeth = target[0] > 0.5
    jaws  = target[1] > 0.5
    iac   = target[2] > 0.5
    rgb[teeth] = np.array([0, 255, 50], dtype=np.uint8)
    rgb[jaws]  = np.maximum(rgb[jaws], np.array([255, 80, 80], dtype=np.uint8))
    rgb[iac]   = np.maximum(rgb[iac], np.array([255, 0, 200], dtype=np.uint8))
    return PIL.Image.fromarray(rgb, "RGB")


# ─────────────────────────────────────────────────────────
# 10.  PER-CASE IMAGE CREATION
# ─────────────────────────────────────────────────────────
def create_case_images(tid: str, vol_img: np.ndarray, vol_lbl: np.ndarray) -> Optional[List[Dict[str, Any]]]:
    if min(vol_img.shape) < 16:
        print(f"  [WARN] {tid}: volume too small {vol_img.shape}")
        return None

    Z = vol_img.shape[0]
    v_r, v_g, v_b = build_rgb_channels(vol_img)
    slab_px = max(12, int(Z * SLAB_FRAC))
    z_positions = sample_z_positions(vol_lbl, NUM_Z_OFFSETS)

    results: List[Dict[str, Any]] = []
    for i, z in enumerate(z_positions):
        img_path       = os.path.join(IMG_DIR, f"{tid}_z{i}.png")
        overlay_path   = os.path.join(DBG_DIR, f"{tid}_z{i}_overlay.png")
        recon_npy_path = os.path.join(RECON_DIR, f"{tid}_z{i}_target.npy")
        recon_png_path = os.path.join(RECON_PNG, f"{tid}_z{i}_target.png")

        y_c, x_c = slab_centroid_yx(vol_lbl, z, slab_px)
        sem = montage_semantics(vol_lbl, z, y_c, x_c)
        if not sem["visible_labels"]:
            continue

        if not os.path.exists(img_path):
            make_montage(v_r, v_g, v_b, z=z, y=y_c, x=x_c).save(img_path)

        if SAVE_DEBUG and not os.path.exists(overlay_path):
            make_debug_overlay(vol_img, vol_lbl, z).save(overlay_path)

        if not os.path.exists(recon_npy_path):
            target = make_recon_target(vol_lbl, z=z, y=y_c, x=x_c, out_hw=RECON_HW)
            np.save(recon_npy_path, target)
        else:
            target = np.load(recon_npy_path).astype(np.float32)

        if not os.path.exists(recon_png_path):
            make_recon_png(target).save(recon_png_path)

        results.append({
            "img_path": img_path,
            "overlay_path": overlay_path,
            "recon_target_npy": recon_npy_path,
            "recon_target_png": recon_png_path,
            "z_idx": i,
            "z_slice": int(z),
            "y_center": int(y_c),
            "x_center": int(x_c),
            "vis_labels": sem["visible_labels"],
            "n_upper": sem["n_upper"],
            "n_lower": sem["n_lower"],
            "upper_teeth_ids": sem["upper_teeth_ids"],
            "lower_teeth_ids": sem["lower_teeth_ids"],
            "n_total_teeth": sem["n_total_teeth"],
            "iac_left": sem["iac_left"],
            "iac_right": sem["iac_right"],
            "present_ids": sem["present_ids"],
            "recon_foreground": {
                "teeth": int(target[0].sum()),
                "jaws": int(target[1].sum()),
                "iac": int(target[2].sum()),
            },
        })

    return results if results else None


# ─────────────────────────────────────────────────────────
# 11.  VQA RECORD BUILDING
# ─────────────────────────────────────────────────────────
_SYSTEM = (
    "You are a specialist dental radiologist interpreting 3D CBCT montage images. "
    "Each montage is a 3×3 grid: rows = axial / coronal / sagittal planes; "
    "columns = center slice / metal-suppressed MIP / slab mean. "
    "Colour channels: Red = bone window, Green = soft-tissue window, "
    "Blue = dense enamel / cortical bone."
)


def _img(p: str) -> Dict[str, str]:
    return {"type": "image", "image": p}


def _txt(t: str) -> Dict[str, str]:
    return {"type": "text", "text": t}


def _base_meta(tid: str, info: Dict[str, Any], task: str, vis: List[str]) -> Dict[str, Any]:
    return {
        "patient_id": tid,
        "z_idx": info["z_idx"],
        "z_slice": info["z_slice"],
        "y_center": info["y_center"],
        "x_center": info["x_center"],
        "task": task,
        "supervision_source": TASK_SRC[task],
        "visible_anatomy": vis,
        "upper_teeth_ids": info["upper_teeth_ids"],
        "lower_teeth_ids": info["lower_teeth_ids"],
        "n_upper": info["n_upper"],
        "n_lower": info["n_lower"],
        "n_total_teeth": info["n_total_teeth"],
        "iac_left": info["iac_left"],
        "iac_right": info["iac_right"],
        "label_space": LABEL_ORDER,
        "source": "ToothFairy2",
        "target_spacing": list(TARGET_SPACING),
        "qwen_resolution": {"min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
        "debug_overlay_path": info["overlay_path"],
        "recon_target_npy": info["recon_target_npy"],
        "recon_target_png": info["recon_target_png"],
        "recon_foreground": info["recon_foreground"],
        "dense_target_source": _SRC_RECON,
    }


def build_classify_record(tid: str, info: Dict[str, Any]) -> Dict[str, Any]:
    vis = info["vis_labels"]
    return {
        "id": f"{tid}_z{info['z_idx']}_{TASK_CLASSIFY}",
        "labels": vis,
        "task": TASK_CLASSIFY,
        "messages": [
            {"role": "user", "content": [
                _img(info["img_path"]),
                _txt(f"{_SYSTEM}\n\n"
                     "Identify which anatomical structures are visible in this CBCT montage.\n"
                     f"Return ONLY a JSON array chosen from {json.dumps(LABEL_ORDER)}. "
                     "Return [] if nothing is visible.")
            ]},
            {"role": "assistant", "content": [_txt(json.dumps(vis, ensure_ascii=False))]},
        ],
        "meta": _base_meta(tid, info, TASK_CLASSIFY, vis),
    }


def _describe_answer(info: Dict[str, Any]) -> str:
    vis = info["vis_labels"]
    n_up = info["n_upper"]
    n_lo = info["n_lower"]
    iac_l = info["iac_left"]
    iac_r = info["iac_right"]
    parts: List[str] = []

    if "Teeth" in vis:
        tooth_parts: List[str] = []
        if n_up > 0:
            tooth_parts.append(f"{n_up} upper-arch tooth/teeth")
        if n_lo > 0:
            tooth_parts.append(f"{n_lo} lower-arch tooth/teeth")
        if tooth_parts:
            parts.append("Teeth are visible: " + " and ".join(tooth_parts) + ".")
        else:
            parts.append("Teeth are visible in the montage.")

    if "Jaws" in vis:
        parts.append("Jaw bone (mandible / maxilla) is present.")

    if "IAC" in vis:
        sides: List[str] = []
        if iac_l:
            sides.append("left")
        if iac_r:
            sides.append("right")
        sides_str = " and ".join(sides) if sides else "one or both sides"
        parts.append(f"The inferior alveolar canal is visible on the {sides_str}.")

    return " ".join(parts) if parts else "No significant anatomical structures identified in this montage."


def build_describe_record(tid: str, info: Dict[str, Any]) -> Dict[str, Any]:
    vis = info["vis_labels"]
    return {
        "id": f"{tid}_z{info['z_idx']}_{TASK_DESCRIBE}",
        "task": TASK_DESCRIBE,
        "messages": [
            {"role": "user", "content": [
                _img(info["img_path"]),
                _txt(f"{_SYSTEM}\n\nDescribe the anatomical structures visible in this CBCT montage.")
            ]},
            {"role": "assistant", "content": [_txt(_describe_answer(info))]},
        ],
        "meta": _base_meta(tid, info, TASK_DESCRIBE, vis),
    }


def build_arch_record(tid: str, info: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if "Teeth" not in info["vis_labels"]:
        return None

    n_up, n_lo = info["n_upper"], info["n_lower"]
    if n_up == 0 and n_lo == 0:
        return None

    if n_up > 0 and n_lo > 0:
        answer = f"Both arches are visible: {n_up} upper-dentition and {n_lo} lower-dentition tooth/teeth present."
        arch_type = "bilateral"
    elif n_up > 0:
        answer = f"Upper dentition only: {n_up} tooth/teeth visible."
        arch_type = "upper"
    else:
        answer = f"Lower dentition only: {n_lo} tooth/teeth visible."
        arch_type = "lower"

    vis = info["vis_labels"]
    meta = {**_base_meta(tid, info, TASK_ARCH, vis), "arch_type": arch_type, "n_upper": n_up, "n_lower": n_lo}
    return {
        "id": f"{tid}_z{info['z_idx']}_{TASK_ARCH}",
        "task": TASK_ARCH,
        "messages": [
            {"role": "user", "content": [
                _img(info["img_path"]),
                _txt(f"{_SYSTEM}\n\nAre upper dentition, lower dentition, or both arches visible in this CBCT montage? Specify how many teeth are present in each arch.")
            ]},
            {"role": "assistant", "content": [_txt(answer)]},
        ],
        "meta": meta,
    }


def build_iac_record(tid: str, info: Dict[str, Any]) -> Dict[str, Any]:
    vis = info["vis_labels"]
    iac_l = info["iac_left"]
    iac_r = info["iac_right"]

    if iac_l and iac_r:
        answer = "Yes — both the left and right inferior alveolar canals are visible in this montage."
        iac_status = "bilateral"
    elif iac_l:
        answer = "Yes — the left inferior alveolar canal is visible."
        iac_status = "left_only"
    elif iac_r:
        answer = "Yes — the right inferior alveolar canal is visible."
        iac_status = "right_only"
    else:
        answer = "No — the inferior alveolar canal is not visible in this montage."
        iac_status = "absent"

    meta = {**_base_meta(tid, info, TASK_IAC, vis), "iac_status": iac_status}
    return {
        "id": f"{tid}_z{info['z_idx']}_{TASK_IAC}",
        "task": TASK_IAC,
        "messages": [
            {"role": "user", "content": [
                _img(info["img_path"]),
                _txt(f"{_SYSTEM}\n\nIs the inferior alveolar canal (IAC) visible in this CBCT montage? If yes, specify which side(s).")
            ]},
            {"role": "assistant", "content": [_txt(answer)]},
        ],
        "meta": meta,
    }


def build_count_record(tid: str, info: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if "Teeth" not in info["vis_labels"]:
        return None

    n_total = info["n_total_teeth"]
    if n_total == 0:
        return None

    answer = f"{n_total} individual tooth/teeth can be identified in this montage: {info['n_upper']} upper and {info['n_lower']} lower."
    vis = info["vis_labels"]
    meta = {**_base_meta(tid, info, TASK_COUNT, vis), "total_teeth": n_total}
    return {
        "id": f"{tid}_z{info['z_idx']}_{TASK_COUNT}",
        "task": TASK_COUNT,
        "messages": [
            {"role": "user", "content": [
                _img(info["img_path"]),
                _txt(f"{_SYSTEM}\n\nHow many individual teeth are visible in this CBCT montage? Specify upper and lower separately.")
            ]},
            {"role": "assistant", "content": [_txt(answer)]},
        ],
        "meta": meta,
    }


_BUILDERS = {
    TASK_CLASSIFY: build_classify_record,
    TASK_DESCRIBE: build_describe_record,
    TASK_ARCH:     build_arch_record,
    TASK_IAC:      build_iac_record,
    TASK_COUNT:    build_count_record,
}


def build_records_for_case(tid: str, slice_infos: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for info in slice_infos:
        for task in ENABLED_TASKS:
            rec = _BUILDERS[task](tid, info)
            if rec is not None:
                records.append(rec)
    return records


# ─────────────────────────────────────────────────────────
# 12.  CLASS IMBALANCE HANDLING
# ─────────────────────────────────────────────────────────
def _target_key(rec: Dict[str, Any]) -> str:
    if rec.get("task") == TASK_CLASSIFY:
        return ",".join(sorted(rec.get("labels", []))) or "None"
    return ",".join(sorted(rec.get("meta", {}).get("visible_anatomy", []))) or "None"


def compute_class_weights(target_counts: Dict[str, int]) -> Dict[str, float]:
    if not target_counts:
        return {}
    total = float(sum(target_counts.values()))
    n_cls = float(len(target_counts))
    raw = {k: total / (n_cls * max(1, v)) for k, v in target_counts.items()}
    mean_w = float(np.mean(list(raw.values())))
    return {k: round(v / mean_w, 4) for k, v in raw.items()}


def compute_sampler_weights(records: List[Dict[str, Any]]) -> List[float]:
    cls_recs = [r for r in records if r.get("task") == TASK_CLASSIFY]
    counts = collections.Counter(_target_key(r) for r in cls_recs)
    cls_w: Dict[Tuple[str, int], float] = {
        (r["meta"]["patient_id"], r["meta"]["z_idx"]): 1.0 / max(1, counts[_target_key(r)])
        for r in cls_recs
    }
    default = 1.0 / max(1, len(counts))
    return [cls_w.get((r["meta"]["patient_id"], r["meta"]["z_idx"]), default) for r in records]


def print_class_report(split_map: Dict[str, List[Dict[str, Any]]]) -> Dict[str, Any]:
    print("\n  Anatomy incidence per split")
    print("  (% = fraction of records in that split carrying each label)")
    print(f"  {'Label':<12} {'Train':>8} {'Val':>8} {'Test':>8}  {'Tr %':>7} {'Va %':>7} {'Te %':>7}")
    print(f"  {'-' * 64}")
    totals = {sp: len(recs) for sp, recs in split_map.items()}
    report: Dict[str, Any] = {}
    for lbl in LABEL_ORDER + ["None"]:
        row: Dict[str, int] = {}
        for sp, recs in split_map.items():
            if lbl == "None":
                row[sp] = sum(1 for r in recs if not r["meta"].get("visible_anatomy"))
            else:
                row[sp] = sum(1 for r in recs if lbl in r["meta"].get("visible_anatomy", []))

        def pct(sp):
            return row[sp] / max(1, totals[sp]) * 100

        print(f"  {lbl:<12} {row.get('train', 0):>8d} {row.get('val', 0):>8d} {row.get('test', 0):>8d}  {pct('train'):>6.1f}% {pct('val'):>6.1f}% {pct('test'):>6.1f}%")
        report[lbl] = row

    print("\n  Anatomy combo distribution (classify records only):")
    for sp, recs in split_map.items():
        cls_recs = [r for r in recs if r.get("task") == TASK_CLASSIFY]
        cnt = collections.Counter(_target_key(r) for r in cls_recs)
        total_sp = len(cls_recs)
        print(f"    {sp} ({total_sp} classify records):")
        for combo, c in sorted(cnt.items(), key=lambda x: -x[1]):
            print(f"      {combo:<30} {c:>5d}  ({c / max(1, total_sp) * 100:.1f}%)")
    print(f"\n  Total records — " + "  ".join(f"{sp}: {n}" for sp, n in totals.items()))
    return report


# ─────────────────────────────────────────────────────────
# 13.  TRAIN / VAL / TEST SPLIT
# ─────────────────────────────────────────────────────────
def _primary_target(recs: List[Dict[str, Any]]) -> str:
    cls_recs = [r for r in recs if r.get("task") == TASK_CLASSIFY]
    if not cls_recs:
        return "None"
    return collections.Counter(_target_key(r) for r in cls_recs).most_common(1)[0][0]


def stratified_split(patient_record_map: Dict[str, List[Dict[str, Any]]], seed: int = SEED, train: float = TRAIN_RATIO, val: float = VAL_RATIO, test: float = TEST_RATIO) -> Tuple[List[str], List[str], List[str]]:
    rng = random.Random(seed)
    buckets: Dict[str, List[str]] = collections.defaultdict(list)
    for pid, recs in patient_record_map.items():
        buckets[_primary_target(recs)].append(pid)

    train_ids: List[str] = []
    val_ids: List[str] = []
    test_ids: List[str] = []

    for key in sorted(buckets):
        pids = list(buckets[key])
        rng.shuffle(pids)
        n = len(pids)
        if n < 3:
            train_ids.extend(pids)
            print(f"  [WARN] bucket '{key}' has {n} patient(s) — all to train.")
            continue
        n_val = max(1, int(round(n * val)))
        n_test = max(1, int(round(n * test)))
        if n - n_val - n_test < 1:
            n_val = max(1, n_val - 1)
        n_train = n - n_val - n_test
        train_ids.extend(pids[:n_train])
        val_ids.extend(pids[n_train:n_train + n_val])
        test_ids.extend(pids[n_train + n_val:])

    rng.shuffle(train_ids)
    rng.shuffle(val_ids)
    rng.shuffle(test_ids)
    return train_ids, val_ids, test_ids


def check_leakage(t: List[str], v: List[str], te: List[str]) -> bool:
    bad = (set(t) & set(v)) | (set(t) & set(te)) | (set(v) & set(te))
    if bad:
        print(f"  [ERROR] Patient leakage: {sorted(bad)[:10]} …")
        return False
    print("  [OK] No patient leakage across splits.")
    return True


def flatten(patient_record_map: Dict[str, List[Dict[str, Any]]], ids: List[str]) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for pid in ids:
        out.extend(patient_record_map.get(pid, []))
    return out


def save_splits(patient_record_map: Dict[str, List[Dict[str, Any]]], train_ids: List[str], val_ids: List[str], test_ids: List[str]) -> Dict[str, str]:
    paths: Dict[str, str] = {}
    for name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
        recs = flatten(patient_record_map, ids)
        p = os.path.join(SPLIT_DIR, f"{name}.jsonl")
        write_jsonl(p, recs)
        paths[name] = p
        if name == "train":
            sw_path = os.path.join(SPLIT_DIR, "train_sampler_weights.json")
            with open(sw_path, "w", encoding="utf-8") as f:
                json.dump({"weights": compute_sampler_weights(recs), "n_records": len(recs)}, f, indent=2)
            paths["sampler_weights"] = sw_path
    return paths


# ─────────────────────────────────────────────────────────
# 14.  MAIN
# ─────────────────────────────────────────────────────────
def main() -> None:
    print("=" * 60)
    print("ToothFairy2 Preprocessing Pipeline")
    print("=" * 60)

    print("\n[1/6] Indexing dataset …")
    dataset = index_dataset(IMAGES_DIR, LABELS_DIR)
    if not dataset:
        print("[ERROR] No matched pairs found. Check TF2_ROOT.")
        return

    print(f"\n[2/6] Processing {len(dataset)} cases …")
    print(f"  Debug overlays → {DBG_DIR}")
    print(f"  Dense recon targets → {RECON_DIR}\n")

    patient_record_map: Dict[str, List[Dict[str, Any]]] = {}
    n_ok, n_failed = 0, 0
    failed_ids: List[str] = []

    for tid, paths in tqdm.tqdm(dataset.items(), desc="  Cases"):
        try:
            vol_img, vol_lbl = load_case(paths["image"], paths["label"])
        except Exception as e:
            print(f"  [ERROR] {tid}: {e}")
            failed_ids.append(tid)
            n_failed += 1
            continue

        slice_infos = create_case_images(tid, vol_img, vol_lbl)
        if not slice_infos:
            print(f"  [WARN] {tid}: no valid slices produced")
            failed_ids.append(tid)
            n_failed += 1
            continue

        records = build_records_for_case(tid, slice_infos)
        if not records:
            failed_ids.append(tid)
            n_failed += 1
            continue

        patient_record_map[tid] = records
        n_ok += 1

    print(f"\n  OK: {n_ok}  |  Failed: {n_failed}")
    if not patient_record_map:
        print("[ERROR] No records produced.")
        return

    all_records = [r for recs in patient_record_map.values() for r in recs]
    write_jsonl(JSONL_SRC, all_records)
    print(f"\n[3/6] Source JSONL: {JSONL_SRC}")
    print(f"  Total records: {len(all_records)}  (~{len(all_records) // max(1, len(patient_record_map))} per patient)")

    print("\n[4/6] Splitting …")
    train_ids, val_ids, test_ids = stratified_split(patient_record_map)
    print(f"  Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")
    check_leakage(train_ids, val_ids, test_ids)
    split_paths = save_splits(patient_record_map, train_ids, val_ids, test_ids)

    print("\n[5/6] Class report …")
    split_map = {
        "train": flatten(patient_record_map, train_ids),
        "val": flatten(patient_record_map, val_ids),
        "test": flatten(patient_record_map, test_ids),
    }
    dist = print_class_report(split_map)

    all_cls_keys = sorted({
        _target_key(r)
        for recs in patient_record_map.values()
        for r in recs
        if r.get("task") == TASK_CLASSIFY
    })
    target_counts = {k: 0 for k in all_cls_keys}
    target_counts.update(collections.Counter(
        _target_key(r)
        for r in split_map["train"]
        if r.get("task") == TASK_CLASSIFY
    ))
    class_weights = compute_class_weights(target_counts)

    print("\n  Class weights (train classify records, for loss function):")
    for combo, w in sorted(class_weights.items(), key=lambda x: -x[1]):
        print(f"    {combo:<35}  weight = {w:.4f}")

    print("\n[6/6] Writing stats …")
    stats = {
        "config_snapshot": CONFIG_SNAPSHOT,
        "n_cases_total": len(dataset),
        "n_cases_ok": n_ok,
        "n_cases_failed": n_failed,
        "n_records_total": len(all_records),
        "n_recon_targets_total": sum(1 for recs in patient_record_map.values() for r in recs if r.get("task") == TASK_CLASSIFY),
        "split_patient_counts": {"train": len(train_ids), "val": len(val_ids), "test": len(test_ids)},
        "split_record_counts": {sp: len(recs) for sp, recs in split_map.items()},
        "anatomy_incidence": dist,
        "class_weights_train": class_weights,
        "paths": split_paths,
        "failed_case_ids": failed_ids,
        "checksums": {
            "source_jsonl": sha256_file(JSONL_SRC),
            "train_jsonl": sha256_file(split_paths["train"]),
            "val_jsonl": sha256_file(split_paths["val"]),
            "test_jsonl": sha256_file(split_paths["test"]),
        },
    }
    with open(STATS_OUT, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(stats), f, indent=2, ensure_ascii=False)

    print(f"[DONE] Stats: {STATS_OUT}")
    print("=" * 60)


if __name__ == "__main__":
    main()

ToothFairy2 Preprocessing Pipeline

[1/6] Indexing dataset …
  Matched pairs: 480

[2/6] Processing 480 cases …
  Debug overlays → /content/drive/MyDrive/Dental_Project/ToothFairy2_Preprocessed/debug_overlays
  Dense recon targets → /content/drive/MyDrive/Dental_Project/ToothFairy2_Preprocessed/recon_targets



  Cases: 100%|██████████| 480/480 [2:34:31<00:00, 19.32s/it]



  OK: 480  |  Failed: 0

[3/6] Source JSONL: /content/drive/MyDrive/Dental_Project/ToothFairy2_Preprocessed/source.jsonl
  Total records: 7200  (~15 per patient)

[4/6] Splitting …
  Train: 336  Val: 72  Test: 72
  [OK] No patient leakage across splits.

[5/6] Class report …

  Anatomy incidence per split
  (% = fraction of records in that split carrying each label)
  Label           Train      Val     Test     Tr %    Va %    Te %
  ----------------------------------------------------------------
  Teeth            5040     1080     1080   100.0%  100.0%  100.0%
  Jaws             3360      720      765    66.7%   66.7%   70.8%
  IAC              4400      935      925    87.3%   86.6%   85.6%
  None                0        0        0     0.0%    0.0%    0.0%

  Anatomy combo distribution (classify records only):
    train (1008 classify records):
      IAC,Jaws,Teeth                   622  (61.7%)
      IAC,Teeth                        258  (25.6%)
      Teeth                       

In [ ]:
import os

# 1. Path to your main folder
folder_path = '/content/drive/MyDrive/Dental_Project/ToothFairy2_Preprocessed'

# 2. Check if the directory exists
if os.path.exists(folder_path):
    # List all items in the root directory
    items = os.listdir(folder_path)
    items.sort()

    print(f"Successfully accessed: {folder_path}")
    print(f"Total items in root: {len(items)}")
    print("-" * 40)

    for item in items:
        item_path = os.path.join(folder_path, item)

        # Check if the item is a folder
        if os.path.isdir(item_path):
            sub_files = os.listdir(item_path)
            sub_files.sort()

            print(f"\n[FOLDER] {item}")
            print(f"   - Total files inside: {len(sub_files)}")

            # Print the first 5 samples from this subfolder
            for sub_file in sub_files[:5]:
                print(f"   --> {sub_file}")

            if len(sub_files) > 5:
                print(f"   ... and {len(sub_files) - 5} more files in this folder.")

        else:
            # It's a file in the root directory
            print(f"[FILE]   {item}")

else:
    print(f"Error: The folder '{folder_path}' does not exist.")
    print("Ensure your Google Drive is mounted correctly.")

Successfully accessed: /content/drive/MyDrive/Dental_Project/ToothFairy2_Preprocessed
Total items in root: 5
----------------------------------------
[FILE]   dataset_stats.json

[FOLDER] debug_overlays
   - Total files inside: 1440
   --> ToothFairy2F_001_z0_overlay.png
   --> ToothFairy2F_001_z1_overlay.png
   --> ToothFairy2F_001_z2_overlay.png
   --> ToothFairy2F_002_z0_overlay.png
   --> ToothFairy2F_002_z1_overlay.png
   ... and 1435 more files in this folder.

[FOLDER] images
   - Total files inside: 1440
   --> ToothFairy2F_001_z0.png
   --> ToothFairy2F_001_z1.png
   --> ToothFairy2F_001_z2.png
   --> ToothFairy2F_002_z0.png
   --> ToothFairy2F_002_z1.png
   ... and 1435 more files in this folder.
[FILE]   source.jsonl

[FOLDER] splits
   - Total files inside: 4
   --> test.jsonl
   --> train.jsonl
   --> train_sampler_weights.json
   --> val.jsonl
